In [2]:
import json
import random
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

# Load the anonymized datasets
with open("/content/friends_AB_anonymized.json", encoding="utf-8") as f:
    scenes_AB = json.load(f)

with open("/content/friends_AC_anonymized.json", encoding="utf-8") as f:
    scenes_AC = json.load(f)

print("A-B segments:", len(scenes_AB))
print("A-C segments:", len(scenes_AC))

# Collect unique episode/conversation IDs across BOTH datasets
all_scenes = scenes_AB + scenes_AC
episode_ids = sorted({
    scene["conversation_id"] for scene in all_scenes
})

train_ids, test_ids = train_test_split(
    episode_ids,
    test_size=0.20,
    random_state=SEED
)

train_ids = set(train_ids)
test_ids = set(test_ids)

# Split both datasets using the same episode-level split
train_AB = [s for s in scenes_AB if s["conversation_id"] in train_ids]
test_AB  = [s for s in scenes_AB if s["conversation_id"] in test_ids]

train_AC = [s for s in scenes_AC if s["conversation_id"] in train_ids]
test_AC  = [s for s in scenes_AC if s["conversation_id"] in test_ids]

print("\nTraining A-B segments:", len(train_AB))
print("Testing A-B segments:", len(test_AB))
print("Training A-C segments:", len(train_AC))
print("Testing A-C segments:", len(test_AC))

# Verify that no episode appears in both splits
assert train_ids.isdisjoint(test_ids)

# Save the split so it can be reused consistently
with open("/content/friends_split.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_episode_ids": sorted(train_ids),
        "test_episode_ids": sorted(test_ids)
    }, f, indent=2)

print("\nEpisode-level split saved.")

A-B segments: 481
A-C segments: 295

Training A-B segments: 383
Testing A-B segments: 98
Training A-C segments: 240
Testing A-C segments: 55

Episode-level split saved.


In [3]:
import random

random.seed(SEED)

MAX_CONTEXT_TURNS = 6
N_DEMOS_PER_PARTNER = 2
N_TEST_PER_PARTNER = 20  # Pilot run; set to None to use all test examples


def make_examples(scenes, max_context_turns=MAX_CONTEXT_TURNS):
    examples = []

    for scene in scenes:
        pair = scene["pair"]
        partner = next((p for p in pair if p != "Person A"), None)

        if partner not in {"Person B", "Person C"}:
            continue

        turns = scene["turns"]

        for i, turn in enumerate(turns):
            # Each example predicts a response from Person A
            if turn["speaker"] != "Person A" or i == 0:
                continue

            context = turns[max(0, i - max_context_turns):i]

            examples.append({
                "conversation_id": scene["conversation_id"],
                "partner": partner,  # Metadata only; don't expose for test prompts
                "context": [
                    {"speaker": t["speaker"], "text": t["text"]}
                    for t in context
                ],
                "gold_response": turn["text"]
            })

    return examples


train_examples = (
    make_examples(train_AB) +
    make_examples(train_AC)
)

test_examples = (
    make_examples(test_AB) +
    make_examples(test_AC)
)

train_AB_examples = [e for e in train_examples if e["partner"] == "Person B"]
train_AC_examples = [e for e in train_examples if e["partner"] == "Person C"]

test_AB_examples = [e for e in test_examples if e["partner"] == "Person B"]
test_AC_examples = [e for e in test_examples if e["partner"] == "Person C"]

print("Training A-B response examples:", len(train_AB_examples))
print("Training A-C response examples:", len(train_AC_examples))
print("Testing A-B response examples:", len(test_AB_examples))
print("Testing A-C response examples:", len(test_AC_examples))

# Pick demonstrations ONLY from training data
assert len(train_AB_examples) >= N_DEMOS_PER_PARTNER
assert len(train_AC_examples) >= N_DEMOS_PER_PARTNER

demo_AB = random.sample(train_AB_examples, N_DEMOS_PER_PARTNER)
demo_AC = random.sample(train_AC_examples, N_DEMOS_PER_PARTNER)

demonstrations = demo_AB + demo_AC
random.shuffle(demonstrations)

# Select a small, balanced pilot test set
if N_TEST_PER_PARTNER is None:
    pilot_AB = test_AB_examples
    pilot_AC = test_AC_examples
else:
    pilot_AB = random.sample(
        test_AB_examples, min(N_TEST_PER_PARTNER, len(test_AB_examples))
    )
    pilot_AC = random.sample(
        test_AC_examples, min(N_TEST_PER_PARTNER, len(test_AC_examples))
    )

pilot_test = pilot_AB + pilot_AC
random.shuffle(pilot_test)

print("\nFew-shot demonstrations:", len(demonstrations))
print("Pilot test examples:", len(pilot_test))

Training A-B response examples: 1548
Training A-C response examples: 681
Testing A-B response examples: 373
Testing A-C response examples: 162

Few-shot demonstrations: 4
Pilot test examples: 40


In [4]:
import re

def format_turns(turns, hide_partner=False):
    lines = []

    for turn in turns:
        speaker = turn["speaker"]
        text = turn["text"]

        if hide_partner:
            # Hide the partner's identity in the speaker label.
            if speaker != "Person A":
                speaker = "[Other Person]"

            # Hide explicit references to either partner in dialogue.
            text = re.sub(
                r"\bPerson [BC]\b",
                "[Other Person]",
                text,
                flags=re.IGNORECASE
            )

        lines.append(f"{speaker}: {text}")

    return "\n".join(lines)



def format_demo(example):
    context = format_turns(example["context"])

    return (
        f'Example — Person A and {example["partner"]}\n'
        f'{context}\n'
        f'Person A: {example["gold_response"]}'
    )



def build_prompt(example, demonstrations):
    demo_text = "\n\n".join(format_demo(d) for d in demonstrations)

    # Hide the test conversation partner's identity.
    test_context = format_turns(
        example["context"],
        hide_partner=True
    )

    return f"""You are generating dialogue as Person A.

Study the examples to learn Person A's general communication
style and how it may vary depending on the person they are
speaking with.

Use the examples to guide your response, while responding
naturally to the current conversation.

Generate exactly one response from Person A.
Do not explain your reasoning or include a speaker label.

EXAMPLES:
{demo_text}

NEW CONVERSATION:
{test_context}

Person A:"""

# Inspect one prompt before loading the model
print(build_prompt(pilot_test[0], demonstrations))

You are generating dialogue as Person A.

Study the examples to learn Person A's general communication
style and how it may vary depending on the person they are
speaking with.

Use the examples to guide your response, while responding
naturally to the current conversation.

Generate exactly one response from Person A.
Do not explain your reasoning or include a speaker label.

EXAMPLES:
Example — Person A and Person C
Person A: Person B said my name up there, I mean, come on, I just can't pretend that didn't happen can I?
Person C: Oh, I-I don't know.
Person A: Person C, what should I do?
Person C: Just uh, do the right thing.
Person A: What?
Person C: Toe the line. Thread the needle. Think outside the box!
Person A: Whoa, wait, listen, I think I'm just gonna talk to Person B about what he think it meant.

Example — Person A and Person C
Person C: Why would I have the keys?
Person A: Aside from the fact that you said you had them?

Example — Person A and Person B
Person B: And I'm fine

In [5]:
!pip -q install -U transformers accelerate bitsandbytes

In [6]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

assert torch.cuda.is_available(), (
    "Please enable a GPU in Runtime > Change runtime type."
)

# Load the model in 4-bit to reduce GPU memory use
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("Model loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded: mistralai/Mistral-7B-Instruct-v0.2
GPU: Tesla T4


In [10]:

def generate_response(example, demonstrations, max_new_tokens=100):
    prompt = build_prompt(example, demonstrations)
    formatted_prompt = f"[INST] {prompt} [/INST]"

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Remove a redundant speaker label if the model adds one
    if response.lower().startswith("person a:"):
        response = response[len("person a:"):].strip()

    return prompt, response


results = []

for i, example in enumerate(pilot_test):
    prompt, generated = generate_response(example, demonstrations)

    results.append({
        "conversation_id": example["conversation_id"],
        "actual_partner": example["partner"],
        "prompt": prompt,
        "context": example["context"],
        "gold_response": example["gold_response"],
        "generated_response": generated
    })

    if (i + 1) % 5 == 0 or i + 1 == len(pilot_test):
        print(f"Generated {i + 1}/{len(pilot_test)} responses")


# Save structured results as JSON
with open(
    "/content/mistral_mixed_fewshot_baseline.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, ensure_ascii=False, indent=2)


# Save prompts and responses as a readable text file
with open(
    "/content/mistral_prompts_and_responses.txt",
    "w",
    encoding="utf-8"
) as f:
    for i, result in enumerate(results, start=1):
        f.write("=" * 80 + "\n")
        f.write(f"EXAMPLE {i}\n")
        f.write(f"Actual partner (evaluation only): {result['actual_partner']}\n")
        f.write("=" * 80 + "\n\n")

        f.write("EXACT PROMPT GIVEN TO MODEL:\n")
        f.write(f"{result['prompt']}\n\n")

        f.write("GOLD RESPONSE:\n")
        f.write(result["gold_response"] + "\n\n")

        f.write("GENERATED RESPONSE:\n")
        f.write(result["generated_response"] + "\n\n\n")

print("Saved JSON: /content/mistral_mixed_fewshot_baseline.json")
print("Saved TXT: /content/mistral_prompts_and_responses.txt")


Generated 5/40 responses
Generated 10/40 responses
Generated 15/40 responses
Generated 20/40 responses
Generated 25/40 responses
Generated 30/40 responses
Generated 35/40 responses
Generated 40/40 responses
Saved JSON: /content/mistral_mixed_fewshot_baseline.json
Saved TXT: /content/mistral_prompts_and_responses.txt
